### Importing Dependencies

In [15]:
import os
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_postgres import PGVector
from tqdm.autonotebook import tqdm
from rich import print

load_dotenv(override=True)

True

### Declaring Embedding Engine

In [16]:
from typing import List
from google import genai
from google.genai import types
from langchain_core.embeddings import Embeddings



class GeminiEmbedder(Embeddings):
    def __init__(self, api_key: str, model: str = "gemini-embedding-2-preview"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        embeddings = self.client.models.embed_content(
        model=self.model,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type="retrieval_document",
         )
        ).embeddings

        return [embedding.values for embedding in embeddings]

    def embed_query(self, text: str) -> List[float]:
        embeddings = self.client.models.embed_content(
        model=self.model,
        contents=text,
        config=types.EmbedContentConfig(
            task_type="retrieval_document",
         )
       ).embeddings
        return embeddings[0].values


embedding_engine = GeminiEmbedder(
    api_key=os.getenv("GOOGLE_API_KEY"),
    model="gemini-embedding-2-preview",
)

### Reading the Knowledge-base Bible

In [17]:
with open("output/livestock_bible_clean.md","r", encoding="utf-8") as f:
    livestock_bible = str(f.read())
# Splitting document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=300, separators=["\n"]
)

splits = text_splitter.split_text(livestock_bible)
print(f"Total splits: {len(splits)}")

Total splits: 877

### Saving to SUPABASE

In [18]:
SUPABASE_PG_CONN_URL = os.getenv("DB_URI")

vector_store = PGVector(
    embeddings=embedding_engine,
    collection_name=os.getenv("COLLECTION_NAME"),
    connection=SUPABASE_PG_CONN_URL,
    use_jsonb=True,
)
print("PGVector Store is loaded.")

# push embedding to collection
batch = 50
for i in tqdm(range(0, len(splits), batch)):
    chunk = splits[i : i + batch]
    vector_store.add_texts(texts=chunk)


PGVector Store is loaded.

100%|██████████| 18/18 [00:57<00:00,  3.19s/it]
